# QualityPhys - Camera Remote Vital Signs Estimator (CRVSE) Project

## Notebook P3-27: VitalVideos-WorldWide Phase-3 Preprocessing

### What this notebook does
Prepares the VitalVideos-WorldWide subset (VV-WW Medium, 300 participants) for Phase 3. The first part works from the ground-truth JSON files alone, before any video is decoded: it audits the cohort and the reference signals, derives recording-level quality flags, and freezes the subject-wise split. The split has to exist before preprocessing because it also decides which iPhone recordings are downloaded (held-out subjects only). The second part, added later, writes the Phase-3 store.

### Dataset
Toye, *VitalVideos-Worldwide: A large and diverse rPPG dataset with rich ground truths* (ICCV 2025 workshop). Each participant was recorded for 60 s, seated and at rest, by two cameras simultaneously:
- **Basler acA1440-220uc** (`<GUID>_1.mp4`): 960x1056, 60 fps, lossless H.264, fixed gain and 12 ms exposure; framing includes face, chest and shoulders.
- **iPhone 15 Pro** (`IMG_xxxx.MOV`): 1080x1920 portrait, 59.94 fps, ProRes; closer framing. Absent for 10 of the 300 participants.

Ground truth, one JSON per participant:
- **BIO** at 500 Hz: PPG (raw BVP sensor), 3-lead ECG (inner wrists, right ankle), respiration belt (waist). Timestamps are `SEQ x 2 ms`; `MS_SINCE_START` is ignored, as the README instructs.
- **CMS** (CMS50D+ pulse oximeter): PPG nominally at 60 Hz but 57-60 Hz in practice, so its waveform is taken to span exactly 60 s, as the documentation requires; HR and SpO2 at 1 Hz.
- Cuff blood pressure before and after; sex, age, Fitzpatrick skin type; a location code and indoor/outdoor environment; free-text notes by the recording staff.

The paper reports ECG and respiration-belt noise in a subset of participants. Publishing any frame is forbidden; no frame is rendered in this notebook.

### Role in Phase 3
- Training corpus candidate (fine-tuning, mixed-corpus training, VitalVideos-only training), with 20% of subjects held out.
- The first Phase-3 data with Fitzpatrick skin type, and so the first skin-type-stratified evaluation, subject to the site confound shown in section 2.
- The only Phase-3 corpus with a respiration reference (belt) and a beat-timing reference (ECG) recorded with the video.

### Decisions taken before this notebook (2026-09-16)
- The Basler MP4 feeds the store. The iPhone MOV breaks the crop contract (in portrait the padded face box was clamped by 12-68% of its side in all 10 subjects checked), so it is kept as raw video for held-out subjects only, for evaluation.
- `frames` is stored at 30 fps, decimated the way the app decimates, so every existing loader reads 160 frames as 5.33 s; `frames_60` keeps the native rate for timing work and augmentation.
- A small downscaled full-frame stream is kept for a motion-based respiration experiment.
- Raw video is deleted once the store is verified; every JSON is kept.

In [1]:
import json
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
from scipy.signal import butter, detrend, filtfilt, find_peaks, resample_poly

# Walk up from the notebook until the repo root is found, identified by the MediaPipe
# asset (the same test NB_P3_25 uses; cwd is the nested notebook folder).
MP_REL = Path('app/live_hr_demo/models/mediapipe/face_landmarker.task')
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / MP_REL).exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError(f'no {MP_REL} in any parent of {Path.cwd()}')

VV_ROOT = Path(r'F:\VitalVideos') # one folder per GUID: <GUID>.json, <GUID>_1.mp4, IMG_xxxx.MOV
VV_META_DIR = Path(r'D:\QualityPhys\VitalsVideo') # local only: tables that carry participant metadata
AUDIT_CSV = VV_META_DIR / 'vitalvideos_reference_audit.csv'
SPLIT_CSV = REPO_ROOT / 'Data' / 'vitalvideos_split.csv' # GUID and split only
MANIFEST_CSV = VV_ROOT / '_meta' / 'remote_manifest.csv' # server listing from the downloader

BIO_FS = 500.0 # research sensors: PPG, 3-lead ECG, respiration belt
SEQ_MS = 2.0 # BIO timestamp = SEQ x 2 ms (README: ignore MS_SINCE_START)
CMS_SPAN_S = 60.0 # CMS50D+ runs at 57-60 Hz; its waveform is taken to span exactly 60 s
PPG_WORK_FS = 50.0 # BIO PPG is decimated 10x before beat detection and spectra
CARDIAC_LOW_HZ, CARDIAC_HIGH_HZ = 0.7, 3.5 # same band as NB_P3_25
CARDIAC_FLOOR_HZ = 0.15 # SQI lobe floor, as NB_P3_25
SQI_WINDOW_S = 30.0 # fixed window for the cross-corpus SQI
HR_MIN_BPM, HR_MAX_BPM = 35.0, 200.0 # physiological limits for beat intervals
HR_AGREE_BPM = 3.0 # PPG is accepted when it agrees with ECG or CMS within this

SPLIT_SEED = 42 # the Phase-3 seed
VAL_FRACTION = 0.2 # the Phase-3 fraction

print('VitalVideos root exists:', VV_ROOT.exists())
print('JSON files:', len(list(VV_ROOT.glob('*/*.json'))) - len(list(VV_ROOT.glob('_*/*.json'))))
print('Split CSV:', SPLIT_CSV, '(exists)' if SPLIT_CSV.exists() else '(not yet written)')

VitalVideos root exists: True
JSON files: 300
Split CSV: D:\code\QualityPhys - CRVSE Project\Data\vitalvideos_split.csv (exists)


## 1. Reading the ground truth
Each JSON is parsed once and reduced to metadata, integrity checks and three independent heart rates; the waveforms themselves are not kept in memory.

- **PPG (BIO, the future `bvp` label):** decimated to 50 Hz, band-passed 0.7-3.5 Hz, and beats detected in whichever polarity gives the steadier beat intervals, since the sensor's sign convention is not documented. Heart rate is 60 over the median beat interval. Beat intervals replace the whole-minute spectral peak used elsewhere: on these recordings the spectral readout disagreed with the ECG by more than 3 bpm in 14 of 300, against 2 for beat intervals, because a heart rate that drifts over the minute spreads its spectral peak.
- **ECG (BIO):** a Pan-Tompkins-style detector (QRS band-pass, squaring, 150 ms integration, refractory peak picking). It is a heart-rate check, not an HRV-grade beat detector.
- **CMS:** the median of the 1 Hz device readings.
- `cardiac_sqi` and `cardiac_sqi_30s` use NB_P3_25's floored-lobe definition, so they compare across the Phase-3 stores. They are recorded, not used as gates.
- **Notes flags** are narrow keyword matches on the staff notes (respiration signal bad, ECG noisy, a noted cardiac condition, CMS PPG bad).

In [2]:
def cardiac_metrics(sig, fps, low=CARDIAC_LOW_HZ, high=CARDIAC_HIGH_HZ,
                    floor_hz=CARDIAC_FLOOR_HZ):
    '''Returns cardiac-band spectral HR (bpm) and a peak-power SQI with a floored lobe.

    Identical to NB_P3_25, so cardiac_sqi and cardiac_sqi_30s are comparable across
    the Phase-3 stores. See that notebook for why the lobe is floored.
    '''
    x = np.asarray(sig, dtype=np.float64)
    if x.size < 16 or not np.all(np.isfinite(x)) or np.std(x) < 1e-9:
        return np.nan, 0.0
    x = (x - x.mean()) * np.hanning(x.size)
    power = np.abs(np.fft.rfft(x)) ** 2
    freqs = np.fft.rfftfreq(x.size, d=1.0 / fps)
    band = (freqs >= low) & (freqs <= high)
    if not band.any() or power[band].sum() <= 0:
        return np.nan, 0.0
    bp = power[band]
    pk = int(np.argmax(bp))
    lobe = max(1, int(round(floor_hz / (freqs[1] - freqs[0]))))
    return (float(freqs[band][pk] * 60.0),
            float(bp[max(0, pk - lobe):pk + lobe + 1].sum() / bp.sum()))


def windowed_metrics(sig, fps, seconds=SQI_WINDOW_S):
    '''Returns median HR and median SQI over fixed-length windows (as NB_P3_25).'''
    n = int(seconds * fps)
    if n > len(sig):
        return cardiac_metrics(sig, fps)
    rows = [cardiac_metrics(sig[i:i + n], fps) for i in range(0, len(sig) - n + 1, n)]
    hr = [h for h, _ in rows if np.isfinite(h)]
    return (float(np.median(hr)) if hr else np.nan,
            float(np.median([s for _, s in rows])))


def beat_rate(peaks, fs):
    '''Returns (HR in bpm, relative beat-interval dispersion, beats) from peak indices.

    Heart rate is 60 over the median beat interval, so an occasional missed or extra
    beat moves it little. Dispersion is the median absolute deviation of the intervals
    over their median: low for a steady rhythm, high for noise or an irregular rhythm.
    '''
    ibi = np.diff(peaks) / fs
    ibi = ibi[(ibi > 60.0 / HR_MAX_BPM) & (ibi < 60.0 / HR_MIN_BPM)]
    if len(ibi) < 10:
        return np.nan, np.nan, len(peaks)
    med = float(np.median(ibi))
    return 60.0 / med, float(np.median(np.abs(ibi - med)) / med), len(peaks)


def ppg_beat_rate(ppg):
    '''Beat-based HR of the 500 Hz BIO PPG.

    The whole-minute spectral peak was the weaker readout here: it disagreed with the
    ECG by more than 3 bpm in 14 of 300 recordings, against 2 for beat intervals, because
    a heart rate that drifts across the minute spreads or splits the spectral peak.
    Peaks are taken in the polarity whose beat intervals are steadier, since the sensor's
    sign convention is not documented.
    '''
    x = resample_poly(detrend(np.asarray(ppg, dtype=np.float64)), 1, int(BIO_FS / PPG_WORK_FS))
    b, a = butter(3, [CARDIAC_LOW_HZ / (PPG_WORK_FS / 2), CARDIAC_HIGH_HZ / (PPG_WORK_FS / 2)],
                  btype='band')
    x = filtfilt(b, a, x)
    x = (x - np.median(x)) / (1.4826 * np.median(np.abs(x - np.median(x))) + 1e-9)
    best = (np.nan, np.nan, 0)
    for sign in (1.0, -1.0):
        peaks, _ = find_peaks(sign * x, distance=int(60.0 / HR_MAX_BPM * PPG_WORK_FS), prominence=0.5)
        hr, disp, n = beat_rate(peaks, PPG_WORK_FS)
        if np.isfinite(hr) and not (np.isfinite(best[1]) and best[1] <= disp):
            best = (hr, disp, n)
    return best


def ecg_beat_rate(ecg):
    '''Beat-based HR of the 500 Hz ECG with a Pan-Tompkins-style detector.

    QRS band-pass (5-15 Hz), squaring and a 150 ms moving integral, then refractory
    peak picking. Squaring makes it polarity-free. It is a heart-rate check, not an HRV
    detector: it double-counts where T waves are tall, which the agreement test below
    catches because the PPG then agrees with the CMS instead.
    '''
    b, a = butter(3, [5.0 / (BIO_FS / 2), 15.0 / (BIO_FS / 2)], btype='band')
    x = filtfilt(b, a, detrend(np.asarray(ecg, dtype=np.float64)))
    k = int(0.15 * BIO_FS)
    integ = np.convolve(x ** 2, np.ones(k) / k, mode='same')
    peaks, _ = find_peaks(integ, distance=int(60.0 / HR_MAX_BPM * BIO_FS),
                          height=integ.mean() + 0.5 * integ.std())
    return beat_rate(peaks, BIO_FS)


BP = re.compile(r'(\d{2,3})\s*/\s*(\d{2,3})')


def parse_bp(value):
    '''Returns (systolic, diastolic) from a "sys/dia" string, or NaNs.'''
    m = BP.search(str(value or ''))
    return (float(m[1]), float(m[2])) if m else (np.nan, np.nan)


def note_flags(text):
    '''Keyword flags from the free-text notes the recording staff wrote.

    Deliberately narrow: "ECG seems inverted" is not flagged as a bad ECG (polarity does
    not affect a squared detector), "fast breathing" or a brief belt disruption is not
    flagged as a bad respiration signal, and a noted heart condition is kept apart from
    signal faults because it is physiology, not a sensor problem.
    '''
    t = str(text or '').lower()
    return dict(
        note_resp_bad=bool(re.search(r'breath\w*[^;]*(bad|noise|untrust)|(bad|no)\s*breath', t)),
        note_ecg_bad=bool(re.search(r'ecg[^;]*(noise|noisy)|(noisy|noise)[^;]*ecg', t)),
        note_cardiac=bool(re.search(r'irregular|heart\s*problem|arrhythm', t)),
        note_cms_bad=bool(re.search(r'cms\s*ppg\s*is\s*bad', t)))


def audit_sample(js_path):
    '''Reads one sample JSON and returns its metadata, integrity and reference metrics.'''
    d = json.loads(js_path.read_text(encoding='utf-8'))
    p, loc = d['participant'], d['location']
    scen = d['scenarios']
    sdata, rec = scen[0]['scenario_data'], scen[0]['recordings']

    bh = rec['BIO'][0]
    bio = np.array(rec['BIO'][1:], dtype=np.float64)
    seq = bio[:, bh.index('SEQ')]
    ppg, ecg = bio[:, bh.index('PPG')], bio[:, bh.index('ECG')]
    ch = rec['CMS'][0]
    cms = np.array(rec['CMS'][1:], dtype=np.float64)
    cms_hr = cms[:, ch.index('hr')]
    cms_hr = cms_hr[(cms_hr >= HR_MIN_BPM) & (cms_hr <= HR_MAX_BPM)]

    ppg_hr, ppg_disp, ppg_beats = ppg_beat_rate(ppg)
    ecg_hr, ecg_disp, ecg_beats = ecg_beat_rate(ecg)
    ppg_work = resample_poly(ppg - ppg.mean(), 1, int(BIO_FS / PPG_WORK_FS))
    _, sqi = cardiac_metrics(ppg_work, PPG_WORK_FS)
    _, sqi_30s = windowed_metrics(ppg_work, PPG_WORK_FS)
    bp_pre, bp_post = parse_bp(rec.get('BPbefore', {}).get('value')), parse_bp(rec.get('BPafter', {}).get('value'))
    notes = f"{p.get('p_notes', '')};{sdata.get('s_notes', '')}"
    return dict(
        guid=d['GUID'], n_scenarios=len(scen), scenario=sdata.get('scenario'),
        sex=p.get('gender'), age=pd.to_numeric(p.get('age'), errors='coerce'),
        fitzpatrick=str(p.get('fitzpatrick') or 'NA'), location=loc.get('location') or 'NA',
        environment=loc.get('environment'), time_field=d.get('time'),
        ext_vid_name=sdata.get('ext_vid_name'), cms_sampling_rate=sdata.get('cms_sampling_rate'),
        bp_pre_sys=bp_pre[0], bp_pre_dia=bp_pre[1], bp_post_sys=bp_post[0], bp_post_dia=bp_post[1],
        bio_rows=len(bio), bio_seq_gaps=int((np.diff(seq) != 1).sum()),
        bio_span_s=float((seq[-1] - seq[0]) * SEQ_MS / 1000.0),
        cms_rows=len(cms), cms_span_s=float((cms[-1, ch.index('time')] - cms[0, ch.index('time')]) / 1000.0),
        hr_ppg=ppg_hr, ppg_ibi_disp=ppg_disp, ppg_beats=ppg_beats,
        hr_ecg=ecg_hr, ecg_ibi_disp=ecg_disp, ecg_beats=ecg_beats,
        hr_cms=float(np.median(cms_hr)) if len(cms_hr) else np.nan,
        cardiac_sqi=sqi, cardiac_sqi_30s=sqi_30s,
        p_notes=p.get('p_notes', ''), s_notes=sdata.get('s_notes', ''), **note_flags(notes))

## 2. Cohort and signal integrity
Tabulates the cohort and checks the waveform blocks: gaps in the BIO sample sequence, BIO and CMS spans, and the CMS sampling rate the device reported.

Location codes are not documented. They are grouped here by how they cluster: three groups of exactly 100 subjects that differ in environment and skin type, consistent with the three collection regions the paper names. The group labels are deliberately neutral. **Skin type and site are confounded:** a difference between Fitzpatrick groups is also a difference between sites, lighting, and outdoor versus indoor recording, and has to be reported as such.

In [3]:
json_paths = sorted(p for p in VV_ROOT.glob('*/*.json') if not p.parent.name.startswith('_'))
audit = pd.DataFrame([audit_sample(p) for p in json_paths])

# Location codes are not documented. They fall into three groups of exactly 100
# subjects that also differ in environment and skin tone, consistent with the three
# collection regions the dataset paper names; the labels here are neutral on purpose.
SITE_GROUPS = {'CH': 'CH*', 'CH2': 'CH*', 'CH3a': 'CH*',
               'CRC': 'CRC/SCH/BET', 'SCH': 'CRC/SCH/BET', 'BET': 'CRC/SCH/BET',
               'MOL': 'MOL/RING', 'RING': 'MOL/RING'}
audit['site_group'] = audit.location.map(SITE_GROUPS).fillna('other')

print(f'{len(audit)} subjects | scenarios {audit.groupby(["n_scenarios", "scenario"]).size().to_dict()}')
print(f'sex {audit.sex.value_counts().to_dict()} | age median {audit.age.median():.0f} '
      f'(range {audit.age.min():.0f}-{audit.age.max():.0f})')
print(f'BIO: SEQ gaps in {int((audit.bio_seq_gaps > 0).sum())} recordings | span '
      f'{audit.bio_span_s.min():.2f}-{audit.bio_span_s.max():.2f} s')
print(f'CMS: span {audit.cms_span_s.min():.2f}-{audit.cms_span_s.max():.2f} s | sampling rate '
      f'{audit.cms_sampling_rate.min():.2f}-{audit.cms_sampling_rate.max():.2f} Hz')
print('\nlocation x Fitzpatrick')
print(pd.crosstab([audit.site_group, audit.location], audit.fitzpatrick, margins=True))
print('\nsite group x environment')
print(pd.crosstab(audit.site_group, audit.environment))

300 subjects | scenarios {(1, 'faceonly'): 300}
sex {'F': 196, 'M': 104} | age median 48 (range 18-88)
BIO: SEQ gaps in 0 recordings | span 59.84-60.29 s
CMS: span 59.90-60.24 s | sampling rate 55.27-60.15 Hz

location x Fitzpatrick
fitzpatrick            1   2   3   4   5   6  All
site_group  location                             
CH*         CH         0   0   0   1  30  49   80
            CH2        0   0   0   0   1   2    3
            CH3a       0   0   0   1  12   4   17
CRC/SCH/BET BET        0   0   2   5   1   0    8
            CRC        0   0   6  35   5   0   46
            SCH        0   0   7  33   6   0   46
MOL/RING    MOL        6  52   1   0   0   1   60
            RING       4  30   4   0   1   1   40
All                   10  82  20  75  56  57  300

site group x environment
environment  indoor  outdoor
site_group                  
CH*               0      100
CRC/SCH/BET      99        1
MOL/RING        100        0


## 3. Heart-rate reference agreement
The BIO PPG becomes the `bvp` label, so it is the signal under test.

- `hr_label_ok` requires the PPG heart rate to agree within 3 bpm with at least one independent reference, ECG or CMS. Requiring both would reject the label for faults in the other sensors, such as ECG noise or T waves counted as beats.
- `ecg_ok`, for later beat-timing work, requires the ECG to agree with the PPG and not to be noted as noisy.

Both flags use the reference signals only and are fixed before any model sees VitalVideos, the rule followed for UBFC-Phys. `cardiac_sqi_30s` is reported but not used as a gate. Respiration quality is not judged here: the belt signal needs its own comparison against ECG- and PPG-derived respiration, in a later section.

In [4]:
audit['d_ppg_ecg'] = (audit.hr_ppg - audit.hr_ecg).abs()
audit['d_ppg_cms'] = (audit.hr_ppg - audit.hr_cms).abs()
audit['d_ecg_cms'] = (audit.hr_ecg - audit.hr_cms).abs()
for col, label in (('d_ppg_ecg', 'PPG vs ECG'), ('d_ppg_cms', 'PPG vs CMS'), ('d_ecg_cms', 'ECG vs CMS')):
    e = audit[col]
    print(f'{label}: median {e.median():.2f} bpm | p99 {e.quantile(0.99):.2f} | '
          f'>{HR_AGREE_BPM:.0f} bpm in {int((e > HR_AGREE_BPM).sum())}')

# The BIO PPG is the training label, so it is the signal under test. It passes when an
# independent reference agrees with it; requiring both would fail it for faults in the
# ECG (noise, T-wave double counting) or the CMS, which are not faults in the label.
audit['hr_label_ok'] = (audit.d_ppg_ecg <= HR_AGREE_BPM) | (audit.d_ppg_cms <= HR_AGREE_BPM)
# ECG usable for beat timing (HRV later): agrees with the PPG and not noted as noisy.
audit['ecg_ok'] = (audit.d_ppg_ecg <= HR_AGREE_BPM) & ~audit.note_ecg_bad

cols = ['guid', 'site_group', 'fitzpatrick', 'hr_ppg', 'hr_ecg', 'hr_cms', 'ppg_ibi_disp',
        'ecg_ibi_disp', 'cardiac_sqi_30s', 'note_ecg_bad', 'note_cardiac']
odd = audit[(audit[['d_ppg_ecg', 'd_ppg_cms', 'd_ecg_cms']] > HR_AGREE_BPM).any(axis=1)]
print(f'\n{len(odd)} recordings where at least one pair disagrees by more than {HR_AGREE_BPM:.0f} bpm:')
print(odd[cols].assign(guid=odd.guid.str[:8]).round(2).to_string(index=False))

print(f'\nhr_label_ok: {int(audit.hr_label_ok.sum())}/{len(audit)} | ecg_ok: {int(audit.ecg_ok.sum())}/{len(audit)}')
print('flags from notes:', {c: int(audit[c].sum()) for c in
                            ('note_resp_bad', 'note_ecg_bad', 'note_cardiac', 'note_cms_bad')})
print('cardiac_sqi_30s: median', round(float(audit.cardiac_sqi_30s.median()), 3),
      '| below 0.40:', audit.loc[audit.cardiac_sqi_30s < 0.40, 'guid'].str[:8].tolist())
print('\nby site group:')
print(audit.groupby('site_group').agg(n=('guid', 'size'), hr_label_ok=('hr_label_ok', 'sum'),
                                      ecg_ok=('ecg_ok', 'sum'), note_resp_bad=('note_resp_bad', 'sum'),
                                      sqi_30s=('cardiac_sqi_30s', 'median')).round(3))

VV_META_DIR.mkdir(parents=True, exist_ok=True)
audit.to_csv(AUDIT_CSV, index=False)
print('\naudit table ->', AUDIT_CSV)

PPG vs ECG: median 0.49 bpm | p99 2.56 | >3 bpm in 2
PPG vs CMS: median 0.71 bpm | p99 5.70 | >3 bpm in 10
ECG vs CMS: median 0.43 bpm | p99 16.77 | >3 bpm in 15

15 recordings where at least one pair disagrees by more than 3 bpm:
    guid  site_group fitzpatrick  hr_ppg  hr_ecg  hr_cms  ppg_ibi_disp  ecg_ibi_disp  cardiac_sqi_30s  note_ecg_bad  note_cardiac
099a7d49    MOL/RING           2   57.69   58.59    52.0          0.08          0.08             0.48         False         False
1adb66cd    MOL/RING           2   65.22   64.66    47.0          0.17          0.22             0.52         False          True
265f65b0    MOL/RING           3   65.22   67.04    64.0          0.04          0.07             0.71         False         False
2ffca8a6    MOL/RING           2   75.00   77.92    74.0          0.03          0.08             0.85         False         False
3f5a86e8    MOL/RING           2   68.18   68.34    72.0          0.02          0.02             0.75         False    

## 4. Frozen subject-wise split
The same rule as the Phase-3 split: subject-wise, `random.Random(42)` over the sorted IDs, 20% validation. Stratification by Fitzpatrick type and location is added: subjects are ordered by (Fitzpatrick, location, shuffled position) and every fifth goes to validation, so each stratum receives its share to within one subject.

Two deliberate differences from the earlier corpora:
- **All 300 subjects are split before any quality filter.** Loaders apply the quality flags inside each split, so a later change to a threshold never moves a subject between train and validation.
- **The split is written once** to `Data/vitalvideos_split.csv`, holding GUID and split only; the audit table with participant metadata stays in `D:\QualityPhys\VitalsVideo\`. Every later run recomputes the split and compares, and raises rather than overwriting if anything differs.

As before, the validation subjects are also the held-out evaluation set. Because they are used for checkpoint selection, held-out figures after training are slightly optimistic.

When VitalVideos joins the pooled Phase-3 corpus, the existing 739-subject assignment must stay untouched. Re-running the pooled shuffle with VitalVideos subjects included would move MCD, DLCN and UBFC subjects between train and validation, including the 8 held-out UBFC subjects.

In [5]:
def assign_split(frame, seed=SPLIT_SEED, fraction=VAL_FRACTION):
    '''Assigns train/val per subject: the Phase-3 seeded shuffle, stratified.

    Subjects are sorted and shuffled with random.Random(seed) exactly as in the Phase-3
    split, then ordered by (fitzpatrick, location, shuffled position) and every k-th
    one (k = 1 / fraction) is taken for validation. Each Fitzpatrick x location stratum
    therefore receives its share to within one subject, and the total is exactly
    round(n * fraction).
    '''
    guids = sorted(frame.guid)
    random.Random(seed).shuffle(guids)
    rank = {g: i for i, g in enumerate(guids)}
    period = int(round(1.0 / fraction))
    ordered = frame.assign(_rank=frame.guid.map(rank)).sort_values(['fitzpatrick', 'location', '_rank'])
    split = pd.Series('train', index=ordered.index)
    split.iloc[period // 2::period] = 'val'
    return split.reindex(frame.index)


audit['split'] = assign_split(audit)
assert (assign_split(audit.sample(frac=1.0, random_state=7)).sort_index() == audit.split).all(), \
    'split depends on row order'

print(audit.split.value_counts().to_dict())
print('\nFitzpatrick x split')
print(pd.crosstab(audit.fitzpatrick, audit.split, margins=True))
print('\nsite group x split')
print(pd.crosstab(audit.site_group, audit.split, margins=True))
print('\nlocation x split')
print(pd.crosstab(audit.location, audit.split))
if MANIFEST_CSV.exists():
    manifest = pd.read_csv(MANIFEST_CSV)
    audit['has_mov'] = audit.guid.isin(manifest.loc[manifest.kind == 'mov', 'guid'])
    print('\niPhone MOV on the server, by split:', audit.groupby('split').has_mov.sum().to_dict())
print('\nbalance:', audit.groupby('split').agg(
    age_median=('age', 'median'), female=('sex', lambda s: int((s == 'F').sum())),
    hr_median=('hr_ppg', 'median'), ecg_ok=('ecg_ok', 'sum'),
    note_resp_bad=('note_resp_bad', 'sum')).to_dict('index'))

frozen = audit[['guid', 'split']].sort_values('guid').reset_index(drop=True)
if SPLIT_CSV.exists():
    on_disk = pd.read_csv(SPLIT_CSV).sort_values('guid').reset_index(drop=True)
    if not on_disk.equals(frozen):
        changed = frozen.merge(on_disk, on='guid', how='outer', suffixes=('_now', '_frozen'))
        changed = changed[changed.split_now != changed.split_frozen]
        raise RuntimeError(f'recomputed split differs from the frozen {SPLIT_CSV.name} in '
                           f'{len(changed)} subjects; the frozen file is the record and is '
                           f'not overwritten')
    print(f'\nsplit matches the frozen {SPLIT_CSV.name}')
else:
    SPLIT_CSV.parent.mkdir(parents=True, exist_ok=True)
    frozen.to_csv(SPLIT_CSV, index=False)
    print(f'\nsplit frozen -> {SPLIT_CSV}')

{'train': 240, 'val': 60}

Fitzpatrick x split
split        train  val  All
fitzpatrick                 
1                8    2   10
2               66   16   82
3               16    4   20
4               60   15   75
5               44   12   56
6               46   11   57
All            240   60  300

site group x split
split        train  val  All
site_group                  
CH*             80   20  100
CRC/SCH/BET     79   21  100
MOL/RING        81   19  100
All            240   60  300

location x split
split     train  val
location            
BET           5    3
CH           64   16
CH2           2    1
CH3a         14    3
CRC          37    9
MOL          49   11
RING         32    8
SCH          37    9

iPhone MOV on the server, by split: {'train': 234, 'val': 56}

balance: {'train': {'age_median': 48.0, 'female': 163, 'hr_median': 76.92307692307692, 'ecg_ok': 238, 'note_resp_bad': 10}, 'val': {'age_median': 48.0, 'female': 33, 'hr_median': 71.42857142857143, 'ecg_ok'

## Findings, part 1 (2026-09-16)
- **Cohort.** 300 participants, 196 female and 104 male, aged 18-88 (median 48), one 60 s `faceonly` recording each. The waveform blocks are intact: no gaps in the BIO sample sequence; BIO spans 59.84-60.29 s and CMS 59.90-60.24 s.
- **Skin type and site are confounded.** The location codes form three groups of exactly 100 subjects: CH* is entirely outdoor and mostly Fitzpatrick 5-6; CRC/SCH/BET is indoor and mostly Fitzpatrick 4; MOL/RING is indoor and mostly Fitzpatrick 1-2. A difference between skin types is also a difference between sites and lighting, and is reported by site group as well.
- **The heart-rate references agree closely.** BIO PPG against ECG: median 0.49 bpm, 99th percentile 2.56. The only two recordings apart by more than 3 bpm are ECG faults (noise in `b6d50b6f`; T waves counted as beats in `e078c9fe`), and in both the PPG agrees with the CMS. The CMS is the least consistent of the three references (above 3 bpm from the ECG in 15 recordings). All 300 PPG labels pass `hr_label_ok`; 296 ECGs pass `ecg_ok`.
- **The SQI is not a noise gate here.** `cardiac_sqi_30s` has a median of 0.81, and its single value below 0.40 belongs to a recording with a noted irregular rhythm rather than a noisy signal.
- **Respiration needs its own check.** The recording staff flagged 12 respiration signals as bad, 10 of them at the CH* sites, so the belt signal is assessed separately before any respiratory-rate work.
- **Split frozen:** 240 train and 60 held-out subjects. Held-out Fitzpatrick 1-6: 2/16/4/15/12/11; site groups 20/21/19; 56 of the 60 held-out subjects have an iPhone MOV. The held-out set is 55% female against 68% in training; sex was not a stratification variable, and the difference is within what chance produces.

## 5. Respiration reference
The respiration belt is the only direct breathing reference in any Phase-3 corpus. The dataset paper reports belt noise in a subset of participants, and the recording staff flagged 12 signals as bad. Staff notes judge how a trace looks rather than whether its rate is right, so the belt is checked against two respiration surrogates recorded by other sensors:
- **ECG-derived respiration (EDR):** the breathing modulation of R-wave amplitude in the limb-lead ECG.
- **PPG pulse amplitude:** the breathing modulation of pulse amplitude in the finger PPG.

All three rates are read from a zero-padded spectrum over the whole minute, within 6-42 breaths/min, refusing peaks on the band edges. `resp_ok` requires the belt rate to agree within 1 breath/min with at least one surrogate. Resting breathing rates cluster in a narrow range, so agreement also occurs by chance; the cell measures that chance level by pairing each belt with the surrogates of shuffled subjects, and reports it beside the result.

Choices made while exploring these 300 recordings (2026-09-16):
- **The belt spectrum rather than breath counting.** Against EDR the spectrum disagreed by a median 0.10 breaths/min, counting by 0.60; a fixed prominence threshold adds or misses shallow breaths.
- **EDR and PPG amplitude rather than the other surrogates tried.** Within 1 breath/min of the belt: EDR 83%, PPG amplitude 73%, ECG RSA 61%, PPG RSA 56%, PPG baseline wander 46%.
- **A 1 breath/min tolerance rather than 2.** It keeps 264 recordings against 270 while nearly halving the chance level (62 against 115 of 300).
- **A 6 breaths/min floor with band-edge refusal.** With a 4.8 floor, slow drift parked 20-27% of the RSA and PPG-amplitude readings on the band edge.

Limitations. `resp_ok` is a recording-level flag over 60 s; window-level respiration labels would need their own check. Surrogates weaken with age (respiratory sinus arrhythmia declines), so part of what the flag measures is the surrogate rather than the belt. Where the belt is refused but EDR and PPG amplitude agree with each other, their mean is kept as `rr_surrogate`; it is recorded for later decisions, not used as a label.

In [6]:
RESP_LOW_HZ, RESP_HIGH_HZ = 0.10, 0.70 # 6-42 breaths/min
RESP_WORK_FS = 10.0 # the belt is decimated 50x before its spectrum
SERIES_FS = 4.0 # beat-to-beat series are resampled onto this grid
RESP_AGREE_BPM = 1.0 # the belt is accepted when an independent surrogate agrees within this
N_SHUFFLES = 200 # shuffled pairings for the chance level


def bandpass(x, fs, low, high, order=2):
    '''Zero-phase Butterworth band-pass.'''
    b, a = butter(order, [low / (fs / 2), high / (fs / 2)], btype='band')
    return filtfilt(b, a, np.asarray(x, dtype=np.float64))


def resp_rate(x, fs, low=RESP_LOW_HZ, high=RESP_HIGH_HZ, pad=8):
    '''Returns the dominant respiratory rate of a series in breaths/min, or NaN.

    A zero-padded FFT with log-parabolic peak refinement, as in the app's heart-rate
    readout. A peak on either band edge is refused: it is slow drift or broadband noise
    rather than breathing, and the low edge is where drift parked several readings
    before the refusal was added.
    '''
    x = np.asarray(x, dtype=np.float64)
    if x.size < 16 or not np.all(np.isfinite(x)) or x.std() < 1e-12:
        return np.nan
    x = (x - x.mean()) * np.hanning(x.size)
    n = x.size * pad
    power = np.abs(np.fft.rfft(x, n)) ** 2
    freqs = np.fft.rfftfreq(n, 1.0 / fs)
    band = np.where((freqs >= low) & (freqs <= high))[0]
    k = int(band[np.argmax(power[band])])
    if k in (band[0], band[-1]):
        return np.nan
    left, centre, right = np.log(power[k - 1:k + 2] + 1e-30)
    denom = left - 2.0 * centre + right
    offset = 0.5 * (left - right) / denom if abs(denom) > 1e-12 else 0.0
    offset = offset if abs(offset) <= 0.5 else 0.0
    return float((k + offset) * (freqs[1] - freqs[0]) * 60.0)


def beat_series_rate(times, values):
    '''Respiratory rate of a beat-sampled series, resampled to SERIES_FS.'''
    ok = np.isfinite(values)
    times, values = np.asarray(times)[ok], np.asarray(values)[ok]
    if len(times) < 10:
        return np.nan
    grid = np.arange(times[0], times[-1], 1.0 / SERIES_FS)
    series = interp1d(times, values, kind='cubic')(grid)
    return resp_rate(detrend(series), SERIES_FS)


def belt_rate(belt):
    '''Respiratory rate of the 500 Hz belt from its spectrum.

    The spectrum is preferred to counting breaths: against ECG-derived respiration it
    agreed to a median 0.10 breaths/min, against 0.60 for breath counting, whose fixed
    prominence threshold adds or misses shallow breaths.
    '''
    x = resample_poly(detrend(np.asarray(belt, dtype=np.float64)), 1, int(BIO_FS / RESP_WORK_FS))
    return resp_rate(bandpass(x, RESP_WORK_FS, 0.05, 1.0), RESP_WORK_FS)


def edr_rate(ecg):
    '''ECG-derived respiration: rate of the breathing modulation of R-wave amplitude.

    Chest movement changes the heart's electrical axis relative to the limb leads, so
    R-wave amplitude rises and falls with each breath. R peaks come from the same
    detector as the heart-rate check, refined to the largest filtered deflection within
    80 ms; an occasional T wave counted as a beat adds noise but rarely a spectral peak.
    '''
    x = bandpass(detrend(np.asarray(ecg, dtype=np.float64)), BIO_FS, 5.0, 15.0, order=3)
    k = int(0.15 * BIO_FS)
    integ = np.convolve(x ** 2, np.ones(k) / k, mode='same')
    peaks, _ = find_peaks(integ, distance=int(60.0 / HR_MAX_BPM * BIO_FS),
                          height=integ.mean() + 0.5 * integ.std())
    w = int(0.08 * BIO_FS)
    r = np.array([max(0, p - w) + int(np.argmax(np.abs(x[max(0, p - w):p + w]))) for p in peaks], dtype=int)
    return beat_series_rate(r / BIO_FS, np.abs(x[r])) if len(r) else np.nan


def ppg_amplitude_rate(ppg):
    '''PPG-derived respiration: rate of the breathing modulation of pulse amplitude.

    Pulse amplitude is each systolic peak minus the preceding trough, in the polarity
    whose beat intervals are steadier (as in ppg_beat_rate).
    '''
    x = resample_poly(detrend(np.asarray(ppg, dtype=np.float64)), 1, int(BIO_FS / PPG_WORK_FS))
    x = bandpass(x, PPG_WORK_FS, CARDIAC_LOW_HZ, CARDIAC_HIGH_HZ, order=3)
    z = (x - np.median(x)) / (1.4826 * np.median(np.abs(x - np.median(x))) + 1e-9)
    best = None
    for sign in (1.0, -1.0):
        peaks, _ = find_peaks(sign * z, distance=int(60.0 / HR_MAX_BPM * PPG_WORK_FS), prominence=0.5)
        ibi = np.diff(peaks) / PPG_WORK_FS
        if len(ibi) >= 10:
            disp = np.median(np.abs(ibi - np.median(ibi))) / np.median(ibi)
            if best is None or disp < best[0]:
                best = (disp, sign, peaks)
    if best is None:
        return np.nan
    _, sign, peaks = best
    troughs, _ = find_peaks(-sign * z, distance=int(60.0 / HR_MAX_BPM * PPG_WORK_FS), prominence=0.5)
    amp = np.full(len(peaks), np.nan)
    for i, p in enumerate(peaks):
        before = troughs[troughs < p]
        if len(before):
            amp[i] = sign * (x[p] - x[before[-1]])
    return beat_series_rate(peaks / PPG_WORK_FS, amp)


def resp_sample(js_path):
    '''Belt, ECG-derived and PPG-derived respiratory rates for one sample JSON.'''
    d = json.loads(js_path.read_text(encoding='utf-8'))
    rec = d['scenarios'][0]['recordings']
    h = rec['BIO'][0]
    bio = np.array(rec['BIO'][1:], dtype=np.float64)
    return dict(guid=d['GUID'], rr_belt=belt_rate(bio[:, h.index('RR')]),
                rr_edr=edr_rate(bio[:, h.index('ECG')]),
                rr_ppg_amp=ppg_amplitude_rate(bio[:, h.index('PPG')]))

In [7]:
resp = pd.DataFrame([resp_sample(p) for p in json_paths])
audit = audit.drop(columns=[c for c in resp.columns if c != 'guid' and c in audit.columns]).merge(resp, on='guid')


def belt_confirmed(belt, edr, ppg_amp, tol=RESP_AGREE_BPM):
    '''True where an independent surrogate agrees with the belt within tol.'''
    with np.errstate(invalid='ignore'):
        return (np.abs(belt - edr) <= tol) | (np.abs(belt - ppg_amp) <= tol)


print('readings (not refused):', {c: int(audit[c].notna().sum()) for c in ('rr_belt', 'rr_edr', 'rr_ppg_amp')})
for col, label in (('rr_edr', 'ECG-derived'), ('rr_ppg_amp', 'PPG amplitude')):
    e = (audit.rr_belt - audit[col]).abs()
    print(f'belt vs {label}: median {e.median():.2f} br/min | within {RESP_AGREE_BPM:.0f}: '
          f'{int((e <= RESP_AGREE_BPM).sum())}/{int(e.notna().sum())}')

belt, edr, ppg_amp = audit.rr_belt.values, audit.rr_edr.values, audit.rr_ppg_amp.values
audit['resp_ok'] = belt_confirmed(belt, edr, ppg_amp)
rng = np.random.default_rng(SPLIT_SEED)
n = len(audit)
chance = np.mean([belt_confirmed(belt, edr[rng.permutation(n)], ppg_amp[rng.permutation(n)]).sum()
                  for _ in range(N_SHUFFLES)])
print(f'\nresp_ok: {int(audit.resp_ok.sum())}/{n} | expected by chance with shuffled surrogates: {chance:.0f}/{n}')
print(f'rr_belt where resp_ok: median {audit.loc[audit.resp_ok, "rr_belt"].median():.1f} br/min, '
      f'range {audit.loc[audit.resp_ok, "rr_belt"].min():.1f}-{audit.loc[audit.resp_ok, "rr_belt"].max():.1f}')

# Where the belt fails but both surrogates agree with each other, the breathing rate is
# probably known even though the belt signal is not usable. Recorded, not used as a label.
with np.errstate(invalid='ignore'):
    surrogates_agree = np.abs(edr - ppg_amp) <= RESP_AGREE_BPM
audit['rr_surrogate'] = np.where(surrogates_agree, (edr + ppg_amp) / 2.0, np.nan)
print(f'belt refused but ECG- and PPG-derived rates agree: {int((~audit.resp_ok & surrogates_agree).sum())}')

audit['age_band'] = pd.cut(audit.age, [0, 40, 60, 120], labels=['<=40', '41-60', '>60'])
for by in ('site_group', 'age_band', 'fitzpatrick', 'split'):
    print(f'\nresp_ok by {by}:', audit.groupby(by, observed=True).resp_ok.agg(['sum', 'size']).T.to_dict())

flagged = audit[audit.note_resp_bad]
print(f'\nstaff-flagged respiration signals ({len(flagged)}):')
print(flagged[['guid', 'site_group', 'rr_belt', 'rr_edr', 'rr_ppg_amp', 'resp_ok', 'rr_surrogate']]
      .assign(guid=flagged.guid.str[:8]).round(1).to_string(index=False))

audit.to_csv(AUDIT_CSV, index=False)
print('\naudit table updated ->', AUDIT_CSV)

readings (not refused): {'rr_belt': 299, 'rr_edr': 296, 'rr_ppg_amp': 268}
belt vs ECG-derived: median 0.10 br/min | within 1: 244/295
belt vs PPG amplitude: median 0.14 br/min | within 1: 196/267

resp_ok: 264/300 | expected by chance with shuffled surrogates: 62/300
rr_belt where resp_ok: median 19.6 br/min, range 6.6-31.1
belt refused but ECG- and PPG-derived rates agree: 14

resp_ok by site_group: {'CH*': {'sum': 90, 'size': 100}, 'CRC/SCH/BET': {'sum': 91, 'size': 100}, 'MOL/RING': {'sum': 83, 'size': 100}}

resp_ok by age_band: {'<=40': {'sum': 85, 'size': 97}, '41-60': {'sum': 130, 'size': 141}, '>60': {'sum': 49, 'size': 62}}

resp_ok by fitzpatrick: {'1': {'sum': 8, 'size': 10}, '2': {'sum': 67, 'size': 82}, '3': {'sum': 15, 'size': 20}, '4': {'sum': 71, 'size': 75}, '5': {'sum': 51, 'size': 56}, '6': {'sum': 52, 'size': 57}}

resp_ok by split: {'train': {'sum': 208, 'size': 240}, 'val': {'sum': 56, 'size': 60}}

staff-flagged respiration signals (12):
    guid  site_group  rr

## Findings, section 5 (2026-09-16)
- **Most belt rates are independently confirmed.** 264 of 300 belt rates agree within 1 breath/min with at least one surrogate, against 62 expected by chance with shuffled surrogates. ECG-derived respiration agrees with the belt to a median 0.10 breaths/min (244 of 295 within 1), PPG pulse amplitude to a median 0.14 (196 of 267).
- **Confirmed rates** span 6.6-31.1 breaths/min, median 19.6.
- **Pass rates** are similar across site groups (CH* 90, CRC/SCH/BET 91, MOL/RING 83 of 100) and lower above 60 years (49 of 62, against 85 of 97 at 40 or younger). That is consistent with weaker respiratory modulation of the cardiac signals at older ages, but these data cannot separate a weaker surrogate from a worse belt. 56 of the 60 held-out subjects have a confirmed belt rate.
- **Staff notes describe the trace, not the rate.** Of the 12 belts the staff flagged, 8 are confirmed by at least one surrogate. In 3 of the 4 refused, the ECG- and PPG-derived rates agree with each other and place the belt 6-16 breaths/min away; in the fourth the surrogates disagree too.
- **`rr_surrogate`** is available for 14 belt-refused recordings. It is kept for later decisions and not used as a label.

# Part 2 - Phase-3 store

### What part 2 does
Writes the VitalVideos subjects into one consolidated HDF5 in the unified Phase-3 schema (`p3_v1`), one group per subject named by GUID, directly in the rechunked uncompressed layout. It reads the audit table and the frozen split from part 1, so after cells 2, 4 and 13 it runs in a fresh kernel.

### Schema (VitalVideos-WW)
| name | source | shape, type | notes |
|------|--------|-------------|-------|
| `frames` | Basler MP4, face crop | [1800, 72, 72, 3] uint8 | 30 fps: every 2nd native frame, as the app decimates 60 fps |
| `frames_60` | Basler MP4, face crop | [3600, 72, 72, 3] uint8 | native 60 fps; `frames` equals `frames_60[::2]` |
| `frames_full` | Basler MP4, whole frame | [600, 132, 120, 3] uint8 | 10 fps: 8x area-downscaled, mean of 6 frames; for a motion-respiration experiment |
| `bvp`, `bvp_60` | BIO PPG | [1800], [3600] float32 | interpolated onto each frame grid; the pulse label |
| `resp` | BIO respiration belt | [1800] float32 | on the 30 fps frame grid |
| `bio_seq`, `bio_ppg`, `bio_ecg`, `bio_resp` | BIO | [~30000] | native 500 Hz; time = `SEQ` x 2 ms |
| `cms_ppg`, `cms_hr`, `cms_spo2` | CMS50D+ | [~3500] float32 | one row per CMS sample; the waveform spans 60 s (`cms_fs` attribute) |

Key attributes:
- `split`, copied from the frozen CSV (which remains the record), and `role` (`train` or `heldout`).
- `fps` 30, `fps_native` 60, `decimation`; the crop box and `box_side_lost`.
- Demographics, blood pressure, and every part-1 flag and reference value (`hr_label_ok`, `ecg_ok`, `resp_ok`, the notes flags, `hr_ppg`, `hr_ecg`, `hr_cms`, `rr_belt`, `rr_edr`, `rr_ppg_amp`, `rr_surrogate`).
- `cardiac_sqi`, `cardiac_sqi_30s`, `bvp_windowed_hr`, and the alignment checks below.

`usable` gates on exposure only: a failure to decode or to find a face raises an error and writes no group. Quality flags are applied by the loaders, not here.

### Crop
The face box and crop are NB_P3_25's functions, unchanged: MediaPipe landmarks on 12 sampled frames, the median box made square in relative coordinates with 0.6 padding, frozen for the recording, clamped to the frame, and INTER_AREA-resized to 72x72. On the 960x1056 Basler frame a relative square is 0.91 as wide as it is tall in pixels, which is also how the app crops such a frame.

### Timing
Native frame k is at k/60 s; BIO sample i is at `SEQ` x 2 ms; both start at zero. A camera survey on 10 subjects found a constant per-subject offset between video and BIO PPG (67-200 ms, pulse transit time plus recording start) and no drift over the minute, so no correction is applied. The offset measured by the alignment check is stored for later timing work.

### Alignment checks (stored, not gates)
POS is computed on the stored `frames` and compared with the stored `bvp`:
- `qc_pos_hr_error`: windowed POS heart rate minus the reference heart rate.
- `qc_pos_bvp_r`, `qc_pos_bvp_lag_s`: the signed correlation at the lag within +/-0.25 s that maximises its magnitude.

In the other Phase-3 stores POS correlates negatively with the label, so a positive median across a whole site group would signal a flipped or misaligned label. The check is only informative where POS itself works: on the dark-skinned outdoor CH* recordings tried so far, POS was weak and single-recording r values were near zero or positive.

### Run
Resumable: groups already in the store are skipped, and the store is flushed after each group. Subjects whose MP4 is still downloading are counted as pending and picked up by the next run. Building a subject takes about 60 s, dominated by decoding, so the full batch (about 4-5 h) runs after the download finishes. `OVERWRITE = True` rebuilds; `MAX_RECORDS` limits a trial run.

In [8]:
import time

import cv2
import h5py
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

MP_MODEL = REPO_ROOT / MP_REL
OUT_DIR = Path(r'D:\QualityPhys\phase 3 datasets')
DATASET_NAME = 'vitalvideos'
OUT_H5 = OUT_DIR / f'{DATASET_NAME}_phase3_rc_none.h5' # rechunked layout, written directly
LOG_PATH = REPO_ROOT / 'Data' / f'processing_log_{DATASET_NAME}_phase3.csv' # no participant metadata

TARGET_SIZE = 72 # committed corpus-wide frame size
FRAMES_PER_CHUNK = 32 # frames per HDF5 chunk, for every frame dataset
CROP_PADDING = 0.6 # expand the face box (include forehead + cheeks), as NB_P3_25
N_DETECT_FRAMES = 12 # frames sampled to compute a stable face box, as NB_P3_25
NATIVE_FPS = 60.0 # Basler MP4
EXPECTED_FRAMES = 3600 # 60 s at 60 fps; any other count is treated as corruption
DECIMATION = 2 # frames = every 2nd native frame -> 30 fps, as the app decimates 60 fps
FULL_BLOCK = 6 # frames_full: mean of 6 native frames -> 10 fps
FULL_DOWNSCALE = 8 # frames_full: 960x1056 -> 120x132
POS_WINDOW_S = 1.6 # POS projection window, as the app
POS_ROI_FRACTION = 0.35 # central fraction of the crop used for the colour trace, as the app
QC_MAX_LAG_S = 0.25 # alignment check searches this far either side (under half a beat below 120 bpm)
FRAME_DARK, FRAME_BRIGHT = 5.0, 250.0 # exposure gate, as NB_P3_25

SCHEMA_VERSION = 'p3_v1'
OVERWRITE = False
MAX_RECORDS = None # set to an int for a trial run

audit_table = pd.read_csv(AUDIT_CSV).set_index('guid')
split_table = pd.read_csv(SPLIT_CSV).set_index('guid')
assert set(audit_table.index) == set(split_table.index), 'audit table and split CSV list different subjects'

print('MediaPipe model exists:', MP_MODEL.exists())
print('Output store:', OUT_H5, '(exists)' if OUT_H5.exists() else '(new)')
print('Audit rows:', len(audit_table), '| split:', split_table.split.value_counts().to_dict())

MediaPipe model exists: True
Output store: D:\QualityPhys\phase 3 datasets\vitalvideos_phase3_rc_none.h5 (exists)
Audit rows: 300 | split: {'train': 240, 'val': 60}


In [9]:
def video_fps(path):
    '''Returns the video frame rate from OpenCV, raising if it cannot be read (as NB_P3_25).'''
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.release()
    if not fps or fps <= 1.0:
        raise RuntimeError(f'unreadable frame rate for {Path(path).name}')
    return float(fps)


def make_landmarker():
    '''Creates a MediaPipe FaceLandmarker in single-image mode.'''
    opts = mp_vision.FaceLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=str(MP_MODEL)),
        running_mode=mp_vision.RunningMode.IMAGE, num_faces=1)
    return mp_vision.FaceLandmarker.create_from_options(opts)


def stable_face_box(frames_rgb, landmarker, padding=CROP_PADDING):
    '''Returns a square relative face box (rx0, ry0, rx1, ry1) as the median detection over
    sample frames, and the number of sample frames with a face. The box is NB_P3_25's.'''
    boxes = []
    for frame in frames_rgb:
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(frame))
        res = landmarker.detect(mp_img)
        if not res.face_landmarks:
            continue
        lm = res.face_landmarks[0]
        xs = np.array([p.x for p in lm])
        ys = np.array([p.y for p in lm])
        boxes.append([xs.min(), ys.min(), xs.max(), ys.max()])
    if not boxes:
        return None, 0
    x0, y0, x1, y1 = np.median(np.array(boxes), axis=0)
    cx, cy = (x0 + x1) / 2.0, (y0 + y1) / 2.0
    half = max(x1 - x0, y1 - y0) * (1.0 + padding) / 2.0
    return (cx - half, cy - half, cx + half, cy + half), len(boxes)


def crop_resize(frame_rgb, box, size):
    '''Crops a frame to a relative square box (clamped to the frame) and resizes to size x size.'''
    h, w = frame_rgb.shape[:2]
    x0 = max(0, int(box[0] * w))
    y0 = max(0, int(box[1] * h))
    x1 = min(w, int(box[2] * w))
    y1 = min(h, int(box[3] * h))
    if x1 <= x0 or y1 <= y0:
        return None
    crop = frame_rgb[y0:y1, x0:x1]
    return cv2.resize(crop, (size, size), interpolation=cv2.INTER_AREA)


def box_side_lost(box):
    '''Fraction of the box side that falls outside the frame, on the worse axis.

    The box is square in relative coordinates, as in every Phase-3 store, so on the
    960x1056 Basler frame it is 0.91 as wide as it is tall in pixels before the resize.
    '''
    lost_x = max(0.0, -box[0]) + max(0.0, box[2] - 1.0)
    lost_y = max(0.0, -box[1]) + max(0.0, box[3] - 1.0)
    return float(max(lost_x / (box[2] - box[0]), lost_y / (box[3] - box[1])))


def extract_streams(video_path, landmarker):
    '''Decodes the MP4 once and returns (frames_60, frames_full, diagnostics).

    frames_60 holds every frame cropped with one frozen box, exactly as NB_P3_25 crops.
    frames_full is the whole frame downscaled FULL_DOWNSCALE times with INTER_AREA and
    averaged over each FULL_BLOCK consecutive frames, so the 10 fps stream carries no
    aliased flicker. Area downscaling is linear, so downscaling before averaging gives
    the same stream as averaging first to within one grey level of rounding (mean 0.10
    on a measured recording) and saves about 6 ms per frame. `frames` (30 fps) is
    frames_60[::DECIMATION], taken by the caller.
    '''
    cap = cv2.VideoCapture(str(video_path))
    n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    sample_at = sorted(set(np.linspace(0, max(n_total - 1, 0), N_DETECT_FRAMES).astype(int)))
    samples = []
    for i in sample_at:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, bgr = cap.read()
        if ok:
            samples.append(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    box, detections = stable_face_box(samples, landmarker) if samples else (None, 0)
    if box is None:
        cap.release()
        raise RuntimeError('face not detected in any sampled frame')

    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    crops, full, acc, height, width = [], [], None, 0, 0
    while True:
        ok, bgr = cap.read()
        if not ok:
            break
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        height, width = rgb.shape[:2]
        c = crop_resize(rgb, box, TARGET_SIZE)
        if c is None:
            cap.release()
            raise RuntimeError('face box falls outside the frame')
        crops.append(c)
        small = cv2.resize(rgb, (width // FULL_DOWNSCALE, height // FULL_DOWNSCALE),
                           interpolation=cv2.INTER_AREA).astype(np.float32)
        acc = small if acc is None else acc + small
        if len(crops) % FULL_BLOCK == 0:
            full.append(np.clip(np.rint(acc / FULL_BLOCK), 0, 255).astype(np.uint8))
            acc = None
    cap.release()
    diagnostics = dict(frame_width=int(width), frame_height=int(height),
                       box_rx0=float(box[0]), box_ry0=float(box[1]),
                       box_rx1=float(box[2]), box_ry1=float(box[3]),
                       box_side_lost=box_side_lost(box), face_detections=int(detections),
                       face_detections_attempted=len(sample_at))
    return np.asarray(crops, dtype=np.uint8), np.asarray(full, dtype=np.uint8), diagnostics


def load_signals(js_path):
    '''Returns the BIO and CMS arrays of one sample JSON, with their time bases in seconds.

    BIO time is SEQ x 2 ms from the first sample (the README's rule). The CMS waveform
    is taken to span exactly CMS_SPAN_S whatever its sample count, as the documentation
    requires, because the device runs below its nominal 60 Hz.
    '''
    d = json.loads(js_path.read_text(encoding='utf-8'))
    rec = d['scenarios'][0]['recordings']
    bh = rec['BIO'][0]
    bio = np.array(rec['BIO'][1:], dtype=np.float64)
    seq = bio[:, bh.index('SEQ')].astype(np.int64)
    ch = rec['CMS'][0]
    cms = np.array(rec['CMS'][1:], dtype=np.float64)
    return dict(bio_seq=seq, bio_t=(seq - seq[0]) * SEQ_MS / 1000.0,
                bio_ppg=bio[:, bh.index('PPG')], bio_ecg=bio[:, bh.index('ECG')],
                bio_resp=bio[:, bh.index('RR')],
                cms_ppg=cms[:, ch.index('ppg')], cms_hr=cms[:, ch.index('hr')],
                cms_spo2=cms[:, ch.index('spo2')], cms_fs=len(cms) / CMS_SPAN_S)


def pos_pulse(frames_u8, fps):
    '''POS pulse (Wang et al., IEEE TBME 2017) from the central crop, as the app computes it.

    Used only for the alignment checks below, never as a label.
    '''
    f = np.asarray(frames_u8)
    h, w = f.shape[1:3]
    dy, dx = int(h * (1 - POS_ROI_FRACTION) / 2), int(w * (1 - POS_ROI_FRACTION) / 2)
    rgb = f[:, dy:h - dy, dx:w - dx, :].reshape(len(f), -1, 3).mean(axis=1)
    length = int(round(POS_WINDOW_S * fps))
    pulse = np.zeros(len(rgb))
    for start in range(0, len(rgb) - length + 1):
        block = rgb[start:start + length]
        norm = block / block.mean(axis=0)
        s1 = norm[:, 1] - norm[:, 2]
        s2 = norm[:, 1] + norm[:, 2] - 2.0 * norm[:, 0]
        combined = s1 + (s1.std() / (s2.std() + 1e-12)) * s2
        pulse[start:start + length] += (combined - combined.mean()) / (combined.std() + 1e-12)
    return bandpass(pulse, fps, CARDIAC_LOW_HZ, CARDIAC_HIGH_HZ, order=4)

def lag_correlations(video, reference, fps, max_lag_s=QC_MAX_LAG_S):
    '''Returns (lags in s, r at each lag) for every whole-frame lag within +/-max_lag_s.

    A positive lag means a feature appears later in the reference than in the video.
    '''
    max_lag = int(round(max_lag_s * fps))
    lags = np.arange(-max_lag, max_lag + 1)
    rs = np.array([np.corrcoef(video[max(0, -lag):len(video) - max(0, lag)],
                               reference[max(0, lag):len(reference) - max(0, -lag)])[0, 1]
                   for lag in lags])
    return lags / fps, rs


def signed_lag(video, reference, fps, max_lag_s=QC_MAX_LAG_S):
    '''Returns (r, lag in s) at the lag within +/-max_lag_s where r is most negative.

    POS and the stored bvp have opposite polarity in every Phase-3 store checked
    (UBFC-rPPG, UBFC-Phys, DLCN, VitalVideos), so the matching alignment is the most
    negative r. Taking the largest |r| instead is ambiguous here: the video trails the
    reference by about 0.1 s, so the opposite-sign peak half a beat later reaches the
    +0.25 s edge at resting heart rates with nearly the same |r|. It won in 30 of the 300
    VitalVideos recordings, 16 of them indoor with POS rates within 3 bpm.
    '''
    lags, rs = lag_correlations(video, reference, fps, max_lag_s)
    if np.all(np.isnan(rs)):
        return float('nan'), float('nan')
    i = int(np.nanargmin(rs))
    return float(rs[i]), float(lags[i])

### Single-recording walkthrough
Builds the first subject whose MP4 has finished downloading, end to end: one decode pass for the three frame streams, the labels and native signals from the JSON, and the alignment checks. It prints the attributes numerically (participant metadata is left out of the printout) and the dataset shapes, and confirms that `frames` equals `frames_60[::2]`. No frame is rendered.

In [10]:
LANDMARKER = make_landmarker()

AUDIT_ATTRS = ('age', 'fitzpatrick', 'location', 'site_group', 'environment',
               'bp_pre_sys', 'bp_pre_dia', 'bp_post_sys', 'bp_post_dia', 'cms_sampling_rate',
               'hr_ppg', 'hr_ecg', 'hr_cms', 'rr_belt', 'rr_edr', 'rr_ppg_amp', 'rr_surrogate',
               'hr_label_ok', 'ecg_ok', 'resp_ok',
               'note_resp_bad', 'note_ecg_bad', 'note_cardiac', 'note_cms_bad')


def plain(value):
    '''Converts a pandas/numpy scalar to a type HDF5 attributes accept (NaN for missing).'''
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return int(value)
    if isinstance(value, (float, np.floating)):
        return float(value)
    return float('nan') if value is None or (isinstance(value, float) and np.isnan(value)) else str(value)


def build_record(guid):
    '''Builds one unified-schema record for a VitalVideos subject from its MP4 and JSON.'''
    t_start = time.perf_counter()
    folder = VV_ROOT / guid
    mp4, js = folder / f'{guid}_1.mp4', folder / f'{guid}.json'
    if not mp4.exists() or not js.exists():
        raise RuntimeError('MP4 or JSON not downloaded')
    fps_native = video_fps(mp4)
    if abs(fps_native - NATIVE_FPS) > 0.01:
        raise RuntimeError(f'container declares {fps_native} fps, expected {NATIVE_FPS}')

    frames_60, frames_full, crop = extract_streams(mp4, LANDMARKER)
    if len(frames_60) != EXPECTED_FRAMES:
        raise RuntimeError(f'{len(frames_60)} frames decoded, expected {EXPECTED_FRAMES}')
    frames = frames_60[::DECIMATION]
    fps = fps_native / DECIMATION

    sig = load_signals(js)
    t60 = np.arange(len(frames_60)) / fps_native
    # np.interp holds its endpoints outside the input range, so a short reference would
    # become a flat stretch of label rather than an error (the NB_P3_25 guard).
    if abs(t60[-1] - sig['bio_t'][-1]) > 1.0:
        raise RuntimeError(f"video spans {t60[-1]:.2f}s but BIO spans {sig['bio_t'][-1]:.2f}s")
    bvp_60 = np.interp(t60, sig['bio_t'], sig['bio_ppg']).astype(np.float32)
    bvp = bvp_60[::DECIMATION]
    resp = np.interp(t60[::DECIMATION], sig['bio_t'], sig['bio_resp']).astype(np.float32)

    bvp_hr, sqi = cardiac_metrics(bvp, fps)
    windowed_hr, sqi_30s = windowed_metrics(bvp, fps, SQI_WINDOW_S)
    pulse = pos_pulse(frames, fps)
    pos_hr, _ = windowed_metrics(pulse, fps, SQI_WINDOW_S)
    r, lag = signed_lag(pulse, bandpass(bvp - bvp.mean(), fps, CARDIAC_LOW_HZ, CARDIAC_HIGH_HZ, order=4), fps)
    brightness = float(np.median([frames[i].mean() for i in np.linspace(0, len(frames) - 1, 9).astype(int)]))
    frame_ok = bool(FRAME_DARK <= brightness <= FRAME_BRIGHT)

    a = audit_table.loc[guid]
    split = str(split_table.loc[guid, 'split'])
    attrs = dict(
        schema_version=SCHEMA_VERSION, dataset='VitalVideos-WW',
        role='train' if split == 'train' else 'heldout', split=split, source_file=mp4.name,
        subject_id=guid, subject_uid=f'vitalvideos_{guid}', gender=str(a['sex']).lower(),
        fps=float(fps), fps_native=float(fps_native), decimation=DECIMATION,
        n_frames=int(len(frames)), n_frames_native=int(len(frames_60)),
        duration_s=float(len(frames_60) / fps_native), frame_size=int(TARGET_SIZE),
        full_fps=float(fps_native / FULL_BLOCK), full_block=FULL_BLOCK, full_downscale=FULL_DOWNSCALE,
        has_frames=True, has_frames_60=True, has_frames_full=True, has_bvp=True, has_resp=True,
        has_ecg=True, has_hr=True, has_spo2=True, has_eda=False,
        bio_fs=BIO_FS, bio_seq_gaps=int((np.diff(sig['bio_seq']) != 1).sum()), cms_fs=float(sig['cms_fs']),
        cardiac_sqi=float(sqi), cardiac_sqi_30s=float(sqi_30s),
        bvp_spectral_hr=float(bvp_hr), bvp_windowed_hr=float(windowed_hr),
        brightness=brightness, frame_ok=frame_ok, usable=frame_ok,
        qc_pos_hr=float(pos_hr), qc_pos_hr_error=float(pos_hr - a['hr_ppg']),
        qc_pos_bvp_r=float(r), qc_pos_bvp_lag_s=float(lag),
        bvp_units_note='BIO research PPG (500 Hz, SEQ x 2 ms) interpolated to frame times',
        resp_units_note='BIO respiration belt (500 Hz) interpolated to the 30 fps frame times',
        seconds_to_build=round(time.perf_counter() - t_start, 1),
        **crop, **{k: plain(a[k]) for k in AUDIT_ATTRS})
    arrays = dict(frames=frames, frames_60=frames_60, frames_full=frames_full,
                  bvp=bvp, bvp_60=bvp_60, resp=resp,
                  bio_seq=sig['bio_seq'].astype(np.int32),
                  bio_ppg=sig['bio_ppg'].astype(np.float32), bio_ecg=sig['bio_ecg'].astype(np.float32),
                  bio_resp=sig['bio_resp'].astype(np.float32),
                  cms_ppg=sig['cms_ppg'].astype(np.float32), cms_hr=sig['cms_hr'].astype(np.float32),
                  cms_spo2=sig['cms_spo2'].astype(np.float32))
    return guid, arrays, attrs


demo = next(g for g in sorted(split_table.index) if (VV_ROOT / g / f'{g}_1.mp4').exists())
rec_id, arrays, attrs = build_record(demo)
print('record:', rec_id[:8])
print(json.dumps({k: (round(v, 3) if isinstance(v, float) else v) for k, v in attrs.items()
                  if k not in AUDIT_ATTRS}, indent=2, default=str))
print({k: (v.shape, str(v.dtype)) for k, v in arrays.items()})
print('frames == frames_60[::2]:', bool(np.array_equal(arrays['frames'], arrays['frames_60'][::DECIMATION])))

record: 01d8e9a0
{
  "schema_version": "p3_v1",
  "dataset": "VitalVideos-WW",
  "role": "train",
  "split": "train",
  "source_file": "01d8e9a045324c078970b530c1e4e924_1.mp4",
  "subject_id": "01d8e9a045324c078970b530c1e4e924",
  "subject_uid": "vitalvideos_01d8e9a045324c078970b530c1e4e924",
  "gender": "f",
  "fps": 30.0,
  "fps_native": 60.0,
  "decimation": 2,
  "n_frames": 1800,
  "n_frames_native": 3600,
  "duration_s": 60.0,
  "frame_size": 72,
  "full_fps": 10.0,
  "full_block": 6,
  "full_downscale": 8,
  "has_frames": true,
  "has_frames_60": true,
  "has_frames_full": true,
  "has_bvp": true,
  "has_resp": true,
  "has_ecg": true,
  "has_hr": true,
  "has_spo2": true,
  "has_eda": false,
  "bio_fs": 500.0,
  "bio_seq_gaps": 0,
  "cms_fs": 59.55,
  "cardiac_sqi": 0.84,
  "cardiac_sqi_30s": 0.831,
  "bvp_spectral_hr": 90.0,
  "bvp_windowed_hr": 92.0,
  "brightness": 66.593,
  "frame_ok": true,
  "usable": true,
  "qc_pos_hr": 62.0,
  "qc_pos_hr_error": -28.909,
  "qc_pos_bvp_r

### Write function and batch processing
Writes each subject as a group in one consolidated HDF5: every frame dataset in `(32, H, W, 3)` chunks, uncompressed (the layout `scripts/rechunk_h5.py` produces for the other stores), and every other dataset unchunked. Only subjects whose MP4 is complete are queued; when the downloader's manifest is present, completeness is checked against the server's file size. Append mode with skip-existing makes the run resumable, and each group is flushed as soon as it is written. The log goes to `Data/processing_log_vitalvideos_phase3.csv` and carries no participant metadata.

In [11]:
def write_group(store, name, arrays, attrs):
    '''Writes one recording as a group: frame datasets chunked along time, then all attributes.'''
    g = store.create_group(name)
    for key, value in arrays.items():
        if key.startswith('frames'):
            # Whole-frame chunks along time let a clip read touch a few chunks and decompress
            # nothing. The length is clamped because h5py rejects chunks longer than the data.
            chunks = (min(FRAMES_PER_CHUNK, len(value)),) + value.shape[1:] if len(value) else None
            g.create_dataset(key, data=value, chunks=chunks)
        else:
            g.create_dataset(key, data=value)
    for key, value in attrs.items():
        g.attrs[key] = value


def mp4_complete(guid, manifest_sizes):
    '''True when the MP4 exists and, if the downloader's manifest is available, has the server's size.'''
    path = VV_ROOT / guid / f'{guid}_1.mp4'
    if not path.exists():
        return False
    expected = manifest_sizes.get(guid)
    return expected is None or path.stat().st_size == expected


LOG_COLS = ('subject_id', 'split', 'source_file', 'n_frames', 'n_frames_native', 'usable', 'frame_ok',
            'face_detections', 'box_side_lost', 'brightness', 'cardiac_sqi_30s', 'bvp_windowed_hr',
            'qc_pos_hr_error', 'qc_pos_bvp_r', 'qc_pos_bvp_lag_s', 'seconds_to_build', 'status', 'error')


def process_all(guids, out_h5, limit=None, overwrite=False):
    '''Writes every subject with a complete MP4 into one consolidated HDF5, one group each.

    Append mode with skip-existing makes the run resumable, and a flush after each group
    keeps finished groups on disk if a later one fails. Subjects whose MP4 is still
    downloading are reported as pending and picked up by the next run.
    '''
    sizes = {}
    if MANIFEST_CSV.exists():
        m = pd.read_csv(MANIFEST_CSV)
        sizes = dict(zip(m.loc[m.kind == 'mp4', 'guid'], m.loc[m.kind == 'mp4', 'remote_size']))
    ready = [g for g in guids if mp4_complete(g, sizes)]
    pending = len(guids) - len(ready)
    if limit:
        ready = ready[:limit]
    rows = []
    t0 = time.perf_counter()
    with h5py.File(out_h5, 'a') as store:
        for i, guid in enumerate(ready, 1):
            if guid in store and not overwrite:
                a = dict(store[guid].attrs)
                a.update(status='skipped', error='')
            else:
                try:
                    _, arrays, attrs = build_record(guid)
                    if guid in store:
                        del store[guid]
                    write_group(store, guid, arrays, attrs)
                    store.flush()
                    a = dict(attrs, status='ok', error='')
                except Exception as exc:
                    a = dict(subject_id=guid, split=str(split_table.loc[guid, 'split']),
                             usable=False, status='error', error=str(exc))
                print(f'   {i}/{len(ready)} {guid[:8]} -> {a["status"]}'
                      f'{" (" + a["error"] + ")" if a["error"] else ""} | '
                      f'{(time.perf_counter() - t0) / 60:.1f} min', flush=True)
            rows.append({c: a.get(c) for c in LOG_COLS})
    return pd.DataFrame(rows), pending


log, pending = process_all(sorted(split_table.index), OUT_H5, limit=MAX_RECORDS, overwrite=OVERWRITE)
log.to_csv(LOG_PATH, index=False)
print('Store:', OUT_H5)
print('ok:', int((log.status == 'ok').sum()), '| skipped:', int((log.status == 'skipped').sum()),
      '| errors:', int((log.status == 'error').sum()), '| MP4 still downloading:', pending)
print('Usable:', int(log.usable.fillna(False).astype(bool).sum()), '/', len(log))
print('Log ->', LOG_PATH)

   1/300 01d8e9a0 -> ok | 0.9 min
   2/300 0216c80c -> ok | 1.8 min
   3/300 03443e46 -> ok | 2.8 min
   4/300 046e64ac -> ok | 3.8 min
   5/300 04dd2594 -> ok | 4.7 min
   6/300 091c53d1 -> ok | 5.7 min
   7/300 0963f270 -> ok | 6.6 min
   8/300 099a7d49 -> ok | 7.6 min
   9/300 09eeac50 -> ok | 8.6 min
   10/300 0ab681d2 -> ok | 9.5 min
   11/300 0c191227 -> ok | 10.6 min
   12/300 0c524b4b -> ok | 11.6 min
   13/300 0dc91832 -> ok | 12.5 min
   14/300 0f1bf857 -> ok | 13.5 min
   15/300 10c24cee -> ok | 14.4 min
   16/300 10e71085 -> ok | 15.3 min
   17/300 12b57fe2 -> ok | 16.3 min
   18/300 1320fa10 -> ok | 17.3 min
   19/300 14626f72 -> ok | 18.3 min
   20/300 17239ec2 -> ok | 19.2 min
   21/300 17b64340 -> ok | 20.1 min
   22/300 19276f2c -> ok | 20.9 min
   23/300 1ab22305 -> ok | 21.9 min
   24/300 1adb66cd -> ok | 22.9 min
   25/300 1af99095 -> ok | 23.9 min
   26/300 1bb88b46 -> ok | 24.8 min
   27/300 1f3ef792 -> ok | 25.8 min
   28/300 1f74a153 -> ok | 26.7 min
   29/300 1

### Output validation
Re-opens the store and checks every group against the schema: the dataset set, the frame and label lengths, and the `split` attribute against the frozen CSV. On a sample of groups it confirms that `frames` equals `frames_60[::2]`. It then summarises usability, face detection, box clamping and brightness, and the alignment checks by site group. These are checks on the stored data, not an evaluation of any method.

In [13]:
with h5py.File(OUT_H5, 'r') as store:
    groups = sorted(store.keys())
    expected_sets = {'frames', 'frames_60', 'frames_full', 'bvp', 'bvp_60', 'resp', 'bio_seq',
                     'bio_ppg', 'bio_ecg', 'bio_resp', 'cms_ppg', 'cms_hr', 'cms_spo2'}
    problems = []
    for g in groups:
        grp = store[g]
        if set(grp.keys()) != expected_sets:
            problems.append((g[:8], 'datasets', sorted(set(grp.keys()) ^ expected_sets)))
        shapes = {k: grp[k].shape for k in ('frames', 'frames_60', 'frames_full', 'bvp', 'bvp_60', 'resp')}
        if (shapes['frames'][0] != shapes['bvp'][0] or shapes['frames_60'][0] != shapes['bvp_60'][0]
                or shapes['resp'][0] != shapes['frames'][0] or shapes['frames'][1:] != (72, 72, 3)):
            problems.append((g[:8], 'shapes', shapes))
        if grp.attrs['split'] != split_table.loc[g, 'split']:
            problems.append((g[:8], 'split attribute differs from the frozen CSV'))
    for g in groups[:: max(1, len(groups) // 5)]:
        grp = store[g]
        if not np.array_equal(grp['frames'][:], grp['frames_60'][::DECIMATION]):
            problems.append((g[:8], 'frames is not frames_60[::2]'))
    qc = pd.DataFrame([{k: store[g].attrs[k] for k in ('site_group', 'split', 'usable', 'box_side_lost',
                                                        'brightness', 'qc_pos_hr_error', 'qc_pos_bvp_r',
                                                        'qc_pos_bvp_lag_s', 'face_detections')}
                       | {'guid': g} for g in groups])

print(f'groups: {len(groups)} | schema problems: {problems or "none"}')
print('usable:', int(qc.usable.sum()), '/', len(qc), '| face detected in all 12 samples:',
      int((qc.face_detections == N_DETECT_FRAMES).sum()), '| box clamped by >2% of its side:',
      int((qc.box_side_lost > 0.02).sum()))
print(f'brightness {qc.brightness.min():.0f}-{qc.brightness.max():.0f}')
# Alignment checks, not an evaluation: POS on the stored crops against the stored label.
# r is the most negative correlation within +/-0.25 s (the Phase-3 polarity); the -0.5
# count is descriptive, not a gate.
print('\nPOS vs bvp by site group:')
print(qc.groupby('site_group').agg(n=('guid', 'size'),
                                   r_median=('qc_pos_bvp_r', 'median'),
                                   r_le_minus_0_5=('qc_pos_bvp_r', lambda s: int((s <= -0.5).sum())),
                                   lag_p25_s=('qc_pos_bvp_lag_s', lambda s: s.quantile(0.25)),
                                   lag_median_s=('qc_pos_bvp_lag_s', 'median'),
                                   lag_p75_s=('qc_pos_bvp_lag_s', lambda s: s.quantile(0.75)),
                                   pos_hr_abs_err_median=('qc_pos_hr_error', lambda s: s.abs().median())).round(3).to_string())

groups: 300 | schema problems: none
usable: 300 / 300 | face detected in all 12 samples: 299 | box clamped by >2% of its side: 33
brightness 34-169

POS vs bvp by site group:
               n  r_median  r_le_minus_0_5  lag_p25_s  lag_median_s  lag_p75_s  pos_hr_abs_err_median
site_group                                                                                           
CH*          100    -0.605              67     -0.100        -0.100     -0.067                  1.079
CRC/SCH/BET  100    -0.850              97     -0.133        -0.100     -0.067                  0.593
MOL/RING     100    -0.844              99     -0.133        -0.133     -0.100                  0.571


## Alignment QC recomputed

In [12]:
def old_rule(lags, rs):
    '''The rule the batch used: first lag with the largest |r| (kept only to recognise old values).'''
    i = int(np.nanargmax(np.abs(rs)))
    return float(rs[i]), float(lags[i])


QC_STORE_MODE = 'r+'  # 'r' reports what would change without writing anything

t0 = time.perf_counter()
rows = []
with h5py.File(OUT_H5, QC_STORE_MODE) as store:
    groups = sorted(store.keys())
    for i, g in enumerate(groups, 1):
        grp = store[g]
        fps = float(grp.attrs['fps'])
        bvp = grp['bvp'][:]
        pulse = pos_pulse(grp['frames'][:], fps)
        reference = bandpass(bvp - bvp.mean(), fps, CARDIAC_LOW_HZ, CARDIAC_HIGH_HZ, order=4)  # as build_record
        lags, rs = lag_correlations(pulse, reference, fps)
        new_r, new_lag = signed_lag(pulse, reference, fps)
        stored = (float(grp.attrs['qc_pos_bvp_r']), float(grp.attrs['qc_pos_bvp_lag_s']))
        # The stored pair must come from this exact recomputation, under either rule, before
        # anything is overwritten; otherwise the frames or labels differ from what was built.
        if stored == (new_r, new_lag):
            state = 'unchanged'
        elif stored == old_rule(lags, rs):
            state = 'updated'
            if QC_STORE_MODE == 'r+':
                grp.attrs['qc_pos_bvp_r'] = new_r
                grp.attrs['qc_pos_bvp_lag_s'] = new_lag
        else:
            state = 'MISMATCH'
        rows.append(dict(subject_id=g, site_group=grp.attrs['site_group'], state=state,
                         old_r=stored[0], old_lag_s=stored[1], qc_pos_bvp_r=new_r,
                         qc_pos_bvp_lag_s=new_lag, max_r=float(np.nanmax(rs)),
                         qc_pos_hr_error=float(grp.attrs['qc_pos_hr_error'])))
        if i % 50 == 0 or i == len(groups):
            print(f'   {i}/{len(groups)} | {(time.perf_counter() - t0) / 60:.1f} min', flush=True)
qc_fix = pd.DataFrame(rows).set_index('subject_id')
print(qc_fix.state.value_counts().to_dict())
assert not (qc_fix.state == 'MISMATCH').any(), 'stored QC does not match a recomputation; nothing more was written'

if QC_STORE_MODE == 'r+':
    # round_trip keeps every other value in the log byte-identical when it is rewritten
    log = pd.read_csv(LOG_PATH, float_precision='round_trip')
    columns = list(log.columns)
    log = log.set_index('subject_id')
    log.loc[qc_fix.index, ['qc_pos_bvp_r', 'qc_pos_bvp_lag_s']] = qc_fix[['qc_pos_bvp_r', 'qc_pos_bvp_lag_s']]
    log.reset_index()[columns].to_csv(LOG_PATH, index=False)
    print('Log updated ->', LOG_PATH)

changed = qc_fix[qc_fix.old_r != qc_fix.qc_pos_bvp_r]
print(f'\nChanged: {len(changed)} recordings (old r was the opposite-sign peak)')
print(changed.groupby('site_group').agg(n=('state', 'size'),
                                        old_lag_s=('old_lag_s', 'median'),
                                        new_r=('qc_pos_bvp_r', 'median'),
                                        new_lag_s=('qc_pos_bvp_lag_s', 'median')).round(3).to_string())

   50/300 | 0.1 min
   100/300 | 0.1 min
   150/300 | 0.2 min
   200/300 | 0.3 min
   250/300 | 0.3 min
   300/300 | 0.4 min
{'unchanged': 270, 'updated': 30}
Log updated -> D:\code\QualityPhys - CRVSE Project\Data\processing_log_vitalvideos_phase3.csv

Changed: 30 recordings (old r was the opposite-sign peak)
              n  old_lag_s  new_r  new_lag_s
site_group                                  
CH*          14      0.233 -0.337     -0.100
CRC/SCH/BET   9      0.233 -0.716     -0.067
MOL/RING      7      0.233 -0.783     -0.167
